In [ ]:
#Loading the trained models
import pandas as pd

sarimax = pd.read_csv("../outputs/sarimax_baseline_forecasts.csv")
random_forest = pd.read_csv("../outputs/random_forest_forecasts.csv")
xgboost = pd.read_csv("../outputs/xgboost_forecasts.csv")
prophet = pd.read_csv("../outputs/prophet_forecasts.csv")

In [2]:
sarimax.shape, random_forest.shape, xgboost.shape, prophet.shape

((1680, 4), (1680, 4), (1680, 4), (1680, 4))

In [3]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

def evaluate_forecast(data):

    actual = data["consumed_tonnes"]
    predicted = data["predicted_tonnes"]

    non_zero = actual > 0

    mape = mean_absolute_percentage_error(
        actual[non_zero],
        predicted[non_zero]
    ) * 100

    rmse = mean_squared_error(
        actual,
        predicted
    ) ** 0.5

    return round(mape, 2), round(rmse, 2)

In [4]:
sarimax_result = evaluate_forecast(sarimax)
rf_result = evaluate_forecast(random_forest)
xgb_result = evaluate_forecast(xgboost)
prophet_result = evaluate_forecast(prophet)

print("SARIMAX:", sarimax_result)
print("Random Forest:", rf_result)
print("XGBoost:", xgb_result)
print("Prophet:", prophet_result)

SARIMAX: (23.79, 9.16)
Random Forest: (19.76, 7.65)
XGBoost: (22.06, 8.12)
Prophet: (23.55, 9.32)


In [5]:
model_comparison = pd.DataFrame({
    "Model": [
        "SARIMAX",
        "Random Forest",
        "XGBoost",
        "Prophet"
    ],
    "MAPE (%)": [
        sarimax_result[0],
        rf_result[0],
        xgb_result[0],
        prophet_result[0]
    ],
    "RMSE (tonnes)": [
        sarimax_result[1],
        rf_result[1],
        xgb_result[1],
        prophet_result[1]
    ]
})

model_comparison

,Model,MAPE (%),RMSE (tonnes)
0,SARIMAX,23.79,9.16
1,Random Forest,19.76,7.65
2,XGBoost,22.06,8.12
3,Prophet,23.55,9.32


- Random Forest is the best performing model

In [7]:
site_results = []

for site in random_forest["site_id"].unique():
    site_data = random_forest[random_forest["site_id"] == site]

    site_mape, site_rmse = evaluate_forecast(site_data)

    site_results.append([site, site_mape, site_rmse])

site_results = pd.DataFrame(
    site_results,
    columns=["site_id", "MAPE (%)", "RMSE (tonnes)"]
)

site_results.sort_values("MAPE (%)").tail()

,site_id,MAPE (%),RMSE (tonnes)
7,SITE_008,41.26,10.80
9,SITE_010,41.84,11.36
24,SITE_025,45.36,11.75
21,SITE_022,53.48,12.28
29,SITE_030,60.43,13.14


te

- Random Forest performance across Sites reveal where the model performed very well and which site did not perform at best

In [8]:
(site_results["MAPE (%)"] <= 15).value_counts()

MAPE (%)
True     16
False    14
Name: count, dtype: int64

- Our model achieve the 15% MAPE across 16 Sites out of 30

In [9]:
model_data = pd.read_csv(
    "../data/processed/cement_forecasting_model_data.csv"
)

site_behavior = model_data[
    ["site_id", "behavior"]
].drop_duplicates()

site_results_with_behavior = site_results.merge(
    site_behavior,
    on="site_id"
)

site_results_with_behavior.groupby("behavior")[
    ["MAPE (%)", "RMSE (tonnes)"]
].mean().round(2)

,MAPE (%),RMSE (tonnes)
behavior,,
aggressive,38.28,10.63
chaotic,7.26,4.40
conservative,0.17,0.12


- Looking the performance of model across the bahviour, just to have sense where it well and perform badly

In [10]:
# Get the aggressive sites
aggressive_sites = site_behavior.loc[
    site_behavior["behavior"] == "aggressive",
    "site_id"
]

# Compare our models on aggressive sites
print("SARIMAX:", evaluate_forecast(
    sarimax[sarimax["site_id"].isin(aggressive_sites)]
))

print("Random Forest:", evaluate_forecast(
    random_forest[random_forest["site_id"].isin(aggressive_sites)]
))

print("XGBoost:", evaluate_forecast(
    xgboost[xgboost["site_id"].isin(aggressive_sites)]
))

print("Prophet:", evaluate_forecast(
    prophet[prophet["site_id"].isin(aggressive_sites)]
))

SARIMAX: (39.82, 12.1)
Random Forest: (38.45, 10.69)
XGBoost: (40.03, 11.32)
Prophet: (40.31, 12.39)


In [12]:
#Saving the models for comparison
model_comparison.to_csv("../outputs/model_comparison.csv", index=False)

site_results_with_behavior.to_csv(
    "../outputs/random_forest_site_evaluation.csv",
    index=False
)

##### Final model: Random Forest

- Overall MAPE: 19.76%

- Overall RMSE: 7.65 tonnes

- Target achieved for 16 of 30 sites

- Aggressive sites remain the main limitation.